In [5]:
import cv2
import numpy as np

def contar_objetos_circulares(frame_bgr, circularidade_min, area_min):
    """
    Conta o número de objetos circulares (de qualquer cor) em um frame.

    Parâmetros:
    - frame_bgr (numpy.ndarray): O frame de vídeo no formato BGR.
    - circularidade_min (float): O valor mínimo do índice de circularidade.
    - area_min (int): A área mínima do contorno (em pixels) para filtrar objetos pequenos.

    Retorna:
    - int: O número de objetos que atendem aos critérios de forma.
    - numpy.ndarray: O frame com os objetos identificados desenhados.
    """
    
    # 1. Converter para escala de cinza
    imagem_cinza = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Aplicar um limiar (threshold) para binarizar a imagem
    # Usamos o método Otsu para encontrar o limiar ideal automaticamente.
    # Isso ajuda a separar objetos do fundo, independentemente da cor.
    _, imagem_binaria = cv2.threshold(imagem_cinza, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Aplicar operações morfológicas para limpar a imagem binária (opcional, mas recomendado)
    kernel = np.ones((5, 5), np.uint8)
    mascara_limpa = cv2.morphologyEx(imagem_binaria, cv2.MORPH_OPEN, kernel)
    mascara_limpa = cv2.morphologyEx(mascara_limpa, cv2.MORPH_CLOSE, kernel)

    # 4. Encontrar contornos na máscara
    contornos, _ = cv2.findContours(mascara_limpa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    objetos_contados = 0
    imagem_resultado = frame_bgr.copy() # Cópia para desenhar o resultado

    # 5. Iterar sobre os contornos e verificar circularidade
    for contorno in contornos:
        area = cv2.contourArea(contorno)

        # 5a. Filtrar contornos muito pequenos (ruído)
        if area < area_min:
            continue

        # 5b. Calcular o perímetro para verificar a circularidade
        perimetro = cv2.arcLength(contorno, True)

        # Evitar divisão por zero
        if perimetro == 0:
            continue

        # Índice de Circularidade: C = 4 * pi * (Area / Perimetro^2)
        # Para um círculo perfeito, C = 1.0.
        circularidade = (4 * np.pi * area) / (perimetro ** 2)

        # 5c. Verificar se o objeto é circular o suficiente
        if circularidade >= circularidade_min:
            objetos_contados += 1
            
            # Opcional: Desenhar o contorno e o círculo envolvente no resultado
            cv2.drawContours(imagem_resultado, [contorno], -1, (0, 255, 0), 2)
            (x, y), raio = cv2.minEnclosingCircle(contorno)
            centro = (int(x), int(y))
            cv2.circle(imagem_resultado, centro, int(raio), (0, 0, 255), 3)
            # Adicionar texto para o número da contagem na imagem
            cv2.putText(imagem_resultado, f"Circularidade: {circularidade:.2f}", 
                        (centro[0] + int(raio) + 10, centro[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return objetos_contados, imagem_resultado




In [6]:
## --- Lógica Principal (Captura e Contagem) ---
# Captura do video
cap = cv2.VideoCapture(0)

# Criar uma janela para o video
cv2.namedWindow('Janela')

# Variável de estado para a contagem total/acumulada
contador_total = 0 
# Contador do frame anterior, usado para verificar se algum objeto sumiu.
contagem_anterior = 0 

while True:
    
    # Capturar frame a frame
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Contar os círculos no frame ATUAL
    contagem_atual, resultado_frame = contar_objetos_circulares(
        frame, 
        circularidade_min=0.85, 
        area_min=100
    )

    # 2. Aplicar a lógica de contagem acumulada (seu novo requisito)
    
    # Caso 1: Se o número de círculos DIMINUIU (o círculo saiu da vista)
    # A lógica aqui é simples: se a contagem atual é menor que a anterior, 
    # subtraímos a diferença (mas você pediu para DECREMENTAR APENAS 1).
   # if contagem_atual < contagem_anterior:
        #contador_total -= 1
    
    # Caso 2: Se temos 2 ou mais círculos NOVOS no view (o número AUMENTOU)
    # Aqui, a lógica é: se a contagem aumentou, ADICIONAMOS a diferença.
    # O requisito "If we have two or more circles in the view it must be added" 
    # é interpretado como: Adicionar o número de círculos que apareceram.
    if contagem_atual > contagem_anterior:
        # Adiciona a diferença, que pode ser 1, 2, ou mais.
        objetos_novos = contagem_atual - contagem_anterior
        contador_total += objetos_novos
    
    # Manter a contagem total no mínimo 0
    if contador_total < 0:
        contador_total = 0

    # 3. Atualizar a contagem anterior para o próximo loop
    contagem_anterior = contagem_atual

    # 4. Mostrar o resultado
    cv2.putText(resultado_frame, f"TOTAL ACUMULADO: {contador_total}", 
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 3)
    cv2.putText(resultado_frame, f"Contagem Frame: {contagem_atual}", 
                (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # Apresentar o frame resultante
    cv2.imshow('Janela', resultado_frame)
    
    # Comando de saída
    if cv2.waitKey(1) & 0xFF == ord('s'):
        break
    
# Quando finalizar, destruir os elementos
cap.release()
cv2.destroyAllWindows()